##### **상품 식재세분류 매핑 모델** 

In [2]:
from init_env import *
from data_loader import load_data
data = load_data(s3=s3, client=client, bucket_nm=bucket_nm)

d = data['d']
df_o = data['df_o']
nm = data['nm']
base = data['base']
mps = data['mps']

===START===
aws access key is done
===함수로딩===
=== 데이터 로드 시작 ===

[1/5] MDM => 식재세분류 로드 중...
✅ d > FS기맵핑(식재세분류): (49678, 6)

[2/5] FW 상품 전체 로드 중...
✅ df_o > FW 상품전체: (227706, 5)

[3/5] 식재 코드-명 매핑 테이블 로드 중...
✅ nm > 식재 코드-명 추출 완료: (988, 2)

[4/5] S3 라벨링 데이터(FW_FSingre_labeling.csv) 로드 중...
✅ base > FW FSingre old 추출 완료: (227649, 16)

[5/5] MPS SKU 데이터 로드 중...
✅ mps > mps 추출: (257201, 30)

=== ✅ 데이터로드 완료 (총 123.74초 소요) ===


In [11]:
# 타깃 데이터 병합/전처리
df_o['gd_no'] = df_o['gd_no'].astype(str)
d ['gd_no'] = d ['gd_no'].astype(str)
base ['gd_no'] = base ['gd_no'].astype(str)
mps ['gd_no'] = mps ['gd_no'].astype(str)

# base 전처리 (괄호제거)
base['gd_nm2'] = [ re.sub(r'[^가-힣]',' ',x.lower()) for x in base['gd_nm'] ]

# check 1 = manual / 2 = model이며 정확도 높아서 정답지로 활용 / 0 = model이며 미스매칭으로 검토 대상
df = pd.merge(df_o, d[['gd_no', 'IGR_CLS_ID_L4', 'IGR_CLS_NM_L4']], how='left', left_on = 'gd_no' , right_on = 'gd_no' ,suffixes=('', '_'))
df['in'] = [ 1 if pd.notna(x) else 0 for x in df['IGR_CLS_ID_L4']]
df = df.merge(mps[['gd_no','mps_cd','mps']], how='left', on='gd_no')
df = df[['gd_no','gd_nm','gd_bd','상품범주id_상세','상품범주명_상세','mps_cd','mps','IGR_CLS_ID_L4','IGR_CLS_NM_L4', 'in']]

df = pd.merge(df, base[['gd_no','IGR_CLS_ID_L4','IGR_CLS_NM_L4','in', 'check']], how='left', on='gd_no', suffixes=('','_'))
# check 값이 float일 수 있으니, df['in'] 컬럼과 맞추기 위해 명시적으로 int로 변환
df.loc[df['check'] == 2, ['IGR_CLS_ID_L4','IGR_CLS_NM_L4','in']] = df.loc[df['check'] == 2, ['IGR_CLS_ID_L4_','IGR_CLS_NM_L4_','check']].assign(check=lambda x: x['check'].astype(int)).values
df = df.drop(columns=['IGR_CLS_ID_L4_', 'IGR_CLS_NM_L4_', 'in_'])
# 상품명 전처리 (괄호제거)
df['gd_nm2'] =  [  re.sub(r'[^가-힣]', ' ', x.lower() )  for x in df['gd_nm'] ]

print("==전처리완료==")

==전처리완료==


In [15]:
df2 = df.copy()
df2[['r111', 'r112', 'r113', 'best_gd_no', 'best_gd_nm']] = pd.NA
target_indices = df2[(df2['in'] == 0) & (df2['mps'] != '제외')].index.tolist()

for i in tqdm(target_indices):
    target_word = df2.loc[i, 'gd_nm2'] # 전처리 상품명
    mps_value = df2.loc[i, 'mps'] # mps명
    category_name = df2.loc[i, '상품범주명_상세'] # 상품세분류명

    # 1️⃣ 후보군 필터링 — MPS 있으면 같은 MPS 내에서, 없으면 같은 상품세분류 내에서
    if pd.notna(mps_value) and mps_value != '제외':
        candidate_df = df2[(df2['mps'] == mps_value) & (df2['IGR_CLS_ID_L4'].notna()) & (df2['mps'] != '제외')]  # ← 안전을 위해 한번 더 체크
    else:
        candidate_df = df2[(df2['상품범주명_상세'] == category_name) & (df2['IGR_CLS_ID_L4'].notna()) &(df2['mps'] != '제외')]  # ← MPS가 '제외'인 후보는 절대 포함하지 않음!

    # 2️⃣ 자기 자신 제외 & 정답지(in == 1,2) + 후보군 리스트 생성
    candidate_df_filtered = candidate_df[~((candidate_df['gd_nm2'] == target_word) & (candidate_df['in'] == 0))]
    candidate_words1 = candidate_df_filtered['gd_nm2'].dropna().tolist()

    # 3️⃣ 유사도 계산 — 가장 유사한 식재세분류와(r111)와 점수(r112) 반환
    if candidate_words1:
        r111, r112 = most_similar_word(target_word, candidate_words1, ngram_range=(1, 2))
        
        # 4️⃣ 유사한 상품 정보(df2에서) → best_gd_no, best_gd_nm, r113(점수) 채우기
        filtered_row = candidate_df_filtered[candidate_df_filtered['gd_nm2'] == r111]
        if not filtered_row.empty:
            # 상품 정보
            df2.loc[i, 'best_gd_no'] = filtered_row['gd_no'].values[0]
            df2.loc[i, 'best_gd_nm'] = filtered_row['gd_nm'].values[0]
            
            # ✅ 분류 정보도 같은 행에서 바로 가져오기!
            df2.loc[i, 'r111'] = filtered_row['IGR_CLS_ID_L4'].values[0]   # 분류 코드
            df2.loc[i, 'r112'] = filtered_row['IGR_CLS_NM_L4'].values[0]   # 분류명
            
            # 유사도 점수
            df2.loc[i, 'r113'] = round(r112, 3)
print("==유사도 처리 완료==")

  4%|▍         | 1824/45350 [02:29<1:00:06, 12.07it/s]

In [ ]:
df2 = df2[['gd_no', 'gd_nm', 'gd_bd', '상품범주id_상세', '상품범주명_상세','mps_cd','mps', 'IGR_CLS_ID_L4',
       'IGR_CLS_NM_L4', 'in', 'r111', 'r112', 'r113','check','best_gd_no','best_gd_nm']]

from io import BytesIO
from datetime import datetime

print("now :", datetime.today())
date = datetime.today().strftime('%y%m%d')
# CSV 내용 생성
csv_content = df2.to_csv(index=False, encoding='utf-8-sig')
# 문자열을 바이트로 인코딩하여 BytesIO로 감싼다
csv_buffer = BytesIO(csv_content.encode('utf-8-sig'))
# 저장
# client.put_object(Bucket=bucket_nm,Key='anal_prj/29.FS_menu_curation/Airflow/output/FW_FSingre_labeling.csv',Body=csv_buffer.getvalue())
client.put_object(Bucket=bucket_nm,Key=f'anal_prj/29.FS_menu_curation/Airflow/output_history/FW_FSingre_labeling_{date}.csv',Body=csv_buffer.getvalue())

print("✅ 파일 저장 in s3 완료")

now : 2025-09-11 13:14:04.091305
✅ 파일 저장 in s3 완료-----------------------------------------------------------------------------
